La app en strealimit primero debe de cargar la Data y subirla a teradata

In [1]:
import teradatasql
import pandas as pd
import polars as pl
 
# Configuración de conexión (ajusta según tu entorno)
TD_HOSTS = ["10.100.232.23", "10.100.232.24"]
TD_USER = "DwhCarga"
TD_PASS = "DN.2020#05"
 
def connect_teradata():
    for host in TD_HOSTS:
        try:
            return teradatasql.connect(
                host=host,
                user=TD_USER,
                password=TD_PASS,
                logmech="TD2",
                encryptdata="true"
            )
        except Exception as e:
            print(f"Fallo conectando a {host}: {e}")
    raise Exception("No se pudo conectar a ningún host")

In [2]:
# Cargar a DataFrame
try:
    with connect_teradata() as conn:
        cursor = conn.cursor()
        cursor.execute("""SELECT * FROM DWH_PRESTAGE.Actualizar_Abril24;""")
        rows = cursor.fetchall()
        cols = [col[0] for col in cursor.description]
        df_1 = pd.DataFrame(rows, columns=cols)
        cursor.close()
except Exception as e:
    print(f"Error al conectar o ejecutar consulta: {e}")
    df_1 = pd.DataFrame()  # DataFrame vacío en caso de error

df_1 = pl.DataFrame(df_1)
print(df_1.shape)
df_1.head(10)

(190, 2)


IdRegistro,DecisionFinal
i64,str
82691,"""P&G"""
83538,"""P&G"""
81878,"""P&G"""
85119,"""P&G"""
85121,"""P&G"""
85122,"""P&G"""
85123,"""P&G"""
85124,"""P&G"""
85125,"""P&G"""


In [3]:
import pyodbc

# Mostrar drivers ODBC disponibles
print("Available ODBC drivers:", pyodbc.drivers())

# Seleccionar el driver instalado
driver = None
for candidate in [
    "ODBC Driver 18 for SQL Server",
    "ODBC Driver 17 for SQL Server",
    "SQL Server Native Client 11.0",
    "SQL Server",
]:
    if candidate in pyodbc.drivers():
        driver = candidate
        break

if driver is None:
    raise RuntimeError(
        "No se encontró un driver SQL Server instalado. "
        "Instala 'ODBC Driver 18 for SQL Server' o 'ODBC Driver 17 for SQL Server'."
    )

# Connection string - replace with your actual server, database, username, and password
conn_str = (
    f"DRIVER={{{driver}}};"
    "SERVER=10.100.181.168;"
    "DATABASE=Mantenimiento_UGI;"
    "UID=USR_APLICACION;"
    "PWD=Interdin2k13;"
    "Encrypt=no;"
    "TrustServerCertificate=yes;"
)

### Establish connection
conn = pyodbc.connect(conn_str)
print("Connection successful! Using driver:", driver)

Available ODBC drivers: ['SQL Server', 'Client Access ODBC Driver (32-bit)', 'iSeries Access ODBC Driver', 'IBM i Access ODBC Driver', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)']
Connection successful! Using driver: SQL Server


In [ ]:
# Create cursor from SQL Server connection
cursor_sql = conn.cursor()

# Execute the query
query = "SELECT * FROM DBO.AUTOSERVICIO_CONSUMOS_NO_RECONOCIDO_DECISION"
cursor_sql.execute(query)

# Get column names
cols = [col[0] for col in cursor_sql.description]
print(f"Expected columns: {len(cols)}")
print(f"Column names: {cols}")

# Fetch all rows
rows_sql = cursor_sql.fetchall()
print(f"Total rows fetched: {len(rows_sql)}")

# Convert pyodbc Row objects to list of dictionaries
data_list = [dict(row) for row in rows_sql]
print(f"First row as dict: {data_list[0] if data_list else 'No data'}")

# Create DataFrame from list of dictionaries (handles Row objects correctly)
df_decision = pd.DataFrame(data_list)

print(f"\nDataFrame shape: {df_decision.shape}")
print(f"DataFrame columns: {list(df_decision.columns)}")
print(f"\nFirst few rows:")
print(df_decision.head())

# Close connection
cursor_sql.close()
conn.close()
print("\nConnection closed.")

ValueError: Shape of passed values is (659, 1), indices imply (659, 2)

In [1]:
import polars as pl
import pandas as pd

In [2]:
df = pl.read_csv(r"C:\Users\adx1152435\Carga_base_anual\base\ISD 2025 UGI.CSV", encoding="latin1", separator=";", infer_schema_length=10000, ignore_errors=True, truncate_ragged_lines=True)
df.shape
print(df.head())

shape: (5, 10)
┌────────────┬────────────┬────────────┬───────────┬───┬───────────┬───────────┬───────────┬───────┐
│ NÚMERO_DE_ ┆ FECHA_DE_T ┆ NÚMERO_DE_ ┆ NOMBRE_DE ┆ … ┆ MONTO_EXE ┆ IMPUESTO_ ┆ MAIL      ┆ MARCA │
│ TRANSACCIÓ ┆ RANSACCIÓN ┆ IDENTIFICA ┆ L_ORDENAN ┆   ┆ NTO       ┆ A_LA_SALI ┆ ---       ┆ ---   │
│ N          ┆ ---        ┆ CIÓN       ┆ TE        ┆   ┆ ---       ┆ DA_DE_DIV ┆ str       ┆ str   │
│ ---        ┆ str        ┆ ---        ┆ ---       ┆   ┆ f64       ┆ ISA…      ┆           ┆       │
│ i64        ┆            ┆ i64        ┆ str       ┆   ┆           ┆ ---       ┆           ┆       │
│            ┆            ┆            ┆           ┆   ┆           ┆ f64       ┆           ┆       │
╞════════════╪════════════╪════════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════╡
│ null       ┆ 04/02/2025 ┆ 100000306  ┆ DARQUEA   ┆ … ┆ 13.98     ┆ 0.0       ┆ mdarqueal ┆ VISA  │
│            ┆            ┆            ┆ LOPEZ     ┆   ┆           ┆        

In [4]:
df.to_pandas().to_parquet('./base/ISD_2025_UGI.parquet')

In [4]:
df2= pl.read_parquet('./base/ISD_2025_UGI.parquet')
print(df2.shape)
df2.head()   

(18868043, 10)


NÚMERO_DE_TRANSACCIÓN,FECHA_DE_TRANSACCIÓN,NÚMERO_DE_IDENTIFICACIÓN,NOMBRE_DEL_ORDENANTE,NOMBRE_DEL_ORDENANTE1,MONTO_TRANSFERIDO,MONTO_EXENTO,IMPUESTO_A_LA_SALIDA_DE_DIVISAS,MAIL,MARCA
f64,str,i64,str,str,f64,f64,f64,str,str
null,"""04/02/2025""",100000306,"""DARQUEA LOPEZ MARCELO SECUNDIN""","""NETFLIXCOM 8""",13.98,13.98,0.0,"""mdarqueal@gmail.com""","""VISA"""
null,"""06/02/2025""",100000306,"""DARQUEA LOPEZ MARCELO SECUNDIN""","""Spotify P33D19857A S""",9.99,9.99,0.0,"""mdarqueal@gmail.com""","""VISA"""
null,"""04/04/2025""",100000306,"""DARQUEA LOPEZ MARCELO SECUNDIN""","""NETFLIXCOM 8""",15.98,15.98,0.0,"""mdarqueal@gmail.com""","""VISA"""
null,"""09/04/2025""",100000306,"""DARQUEA LOPEZ MARCELO SECUNDIN""","""Spotify P35B1A4D44 S""",13.48,13.48,0.0,"""mdarqueal@gmail.com""","""VISA"""
null,"""04/06/2025""",100000306,"""DARQUEA LOPEZ MARCELO SECUNDIN""","""NETFLIXCOM 8""",15.98,15.98,0.0,"""mdarqueal@gmail.com""","""VISA"""


In [6]:
df2.schema

Schema([('NÚMERO_DE_TRANSACCIÓN', Float64),
        ('FECHA_DE_TRANSACCIÓN', String),
        ('NÚMERO_DE_IDENTIFICACIÓN', Int64),
        ('NOMBRE_DEL_ORDENANTE', String),
        ('NOMBRE_DEL_ORDENANTE1', String),
        ('MONTO_TRANSFERIDO', Float64),
        ('MONTO_EXENTO', Float64),
        ('IMPUESTO_A_LA_SALIDA_DE_DIVISAS', Float64),
        ('MAIL', String),
        ('MARCA', String)])

In [1]:
import teradatasql
import pandas as pd
import polars as pl
 
# Configuración de conexión (ajusta según tu entorno)
TD_HOSTS = ["10.100.232.23", "10.100.232.24"]
TD_USER = "DwhCarga"
TD_PASS = "DN.2020#05"
 
def connect_teradata():
    for host in TD_HOSTS:
        try:
            return teradatasql.connect(
                host=host,
                user=TD_USER,
                password=TD_PASS,
                logmech="TD2",
                encryptdata="true"
            )
        except Exception as e:
            print(f"Fallo conectando a {host}: {e}")
    raise Exception("No se pudo conectar a ningún host")

In [ ]:
# Cargar a DataFrame
try:
    with connect_teradata() as conn:
        cursor = conn.cursor()
        cursor.execute("""SELECT * FROM DWH_PRESTAGE.Actualizar_Abril24;""")
        rows = cursor.fetchall()
        cols = [col[0] for col in cursor.description]
        df_1 = pd.DataFrame(rows, columns=cols)
        cursor.close()
except Exception as e:
    print(f"Error al conectar o ejecutar consulta: {e}")
    df_1 = pd.DataFrame()  # DataFrame vacío en caso de error

df_1 = pl.DataFrame(df_1)
print(df_1.shape)
df_1.head(10)

(190, 2)


IdRegistro,DecisionFinal
i64,str
82691,"""P&G"""
83538,"""P&G"""
81878,"""P&G"""
85119,"""P&G"""
85121,"""P&G"""
85122,"""P&G"""
85123,"""P&G"""
85124,"""P&G"""
85125,"""P&G"""
